# KPI 11 — Lead Conversion Funnel

**requirements.md (Lead Quality & Sales Funnel, KPI 11):** "Lead Conversion Funnel: Lead source → orders; conversion rates by campaign/channel."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_11_lead_conversion_funnel` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cells below once (or after any change to the fact definitions).

**Definitions (state formulas):**

* **Grain:** one row per (**source**, **campaign_name**) — 8 campaigns across 7 sources (`utm_source` / `utm_medium` / `utm_campaign` carried as attributes; verified 1:1 with campaign, so grouping the view by `utm_medium` answers the channel question).
* **Funnel counts:** `new_` / `contacted_` / `qualified_` / `proposal_` / `won_` / `lost_lead_count` — the 6 fact_leads statuses; sum exactly to `lead_count` on every row. Conversion rates over `lead_count`: `qualified_rate_pct`, `proposal_rate_pct`, `won_rate_pct`, `lost_rate_pct` = ROUND(100 × count / lead_count, 2). Stage conversion `won_over_qualified_pct` = ROUND(100 × won / qualified, 2).
* **Order attribution (company bridge):** leads carry no company_id — `fact_leads.company_sk` is the case-insensitive company_name match against dim_companies (built into the fact; all 144 matched leads are Won; 17 Won leads are unmatched — companies not yet in dim_companies — with no attributable orders, surfaced by `won_matched_lead_count` vs `won_lead_count`). Matched Won leads → `fact_order_items.company_sk` → orders/revenue at **company grain, full order history, no date cut-off** (revenue measured on the KPI 1 bases below; gross corrected 2026-11-20 — the pre-fix $10,912,463.13, like the fact_leads build verification, summed invalid-date lines). Revenue bases = KPI 1: gross = all statuses + `order_date_valid`; net = excluding `cancelled` + `order_date_valid`. Each matched company is attributed to exactly ONE campaign (verified: 144 Won+matched leads = 144 distinct companies, 0 in >1 campaign), so attributed totals **sum exactly** across the 8 rows.
* **Measures:** `won_matched_lead_count`, `won_leads_with_orders_count`, `attributed_order_count`, `attributed_net_order_count`, `attributed_gross_revenue`, `attributed_net_revenue`, `attributed_net_revenue_per_won_lead`; rollups `lead_share_pct` + `attributed_net_revenue_share_pct` (per-row ROUND, sum ≈ 100.00) and `attributed_net_revenue_rank` = ROW_NUMBER by attributed net revenue DESC (tie-break source, campaign — deterministic).
* **Verified:** 8 rows = 8 distinct (source, campaign) pairs; 1,098 leads = 285 New + 237 Contacted + 138 Qualified + 94 Proposal + 161 Won + 183 Lost; 144 won+matched / 140 with orders; 3,530 attributed orders (3,147 net) / 10,551,748.55 gross / 9,766,828.34 net — net exact match to the fact_leads build verification; gross = KPI 1 valid-date basis (fix 2026-11-20, exact tie with kpi_12); 8/8 full matches via independent recompute.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_11_lead_conversion_funnel
TBLPROPERTIES (
  'comment' = 'KPI 11 - Lead conversion funnel (requirements.md: Lead Conversion Funnel: Lead source -> orders; conversion rates by campaign/channel). Grain: one row per (source, campaign_name) - 8 campaigns across 7 sources; utm_source/utm_medium/utm_campaign carried as attributes (verified 1:1 with campaign). Funnel counts sum exactly to lead_count: new/contacted/qualified/proposal/won/lost_lead_count; rates over lead_count: qualified_rate_pct, proposal_rate_pct, won_rate_pct, lost_rate_pct = ROUND(100*count/lead_count,2); stage conversion won_over_qualified_pct = ROUND(100*won/qualified,2). Order attribution: Won leads matched to dim_companies via company_sk (company_name bridge built into fact_leads; all 144 matched leads are Won; 17 Won leads unmatched - companies not yet in dim - have no attributable orders), then company_sk -> fact_order_items orders/revenue at COMPANY grain, full order history, no date cut-off (matches fact_leads build verification). KPI 1 revenue bases: gross = all statuses + order_date_valid; net = excl cancelled + order_date_valid. Each matched company attributed to exactly ONE campaign (verified: 144 Won+matched leads = 144 distinct companies, 0 in >1 campaign) so attributed totals sum exactly across rows. Measures: won_matched_lead_count, won_leads_with_orders_count, attributed_order_count, attributed_net_order_count, attributed_gross_revenue, attributed_net_revenue, attributed_net_revenue_per_won_lead; rollups lead_share_pct + attributed_net_revenue_share_pct (per-row ROUND, sum ~100.00) and attributed_net_revenue_rank = ROW_NUMBER by attributed net revenue DESC (tie-break source, campaign - deterministic). Verified: 8 rows; 1,098 leads = 285 New + 237 Contacted + 138 Qualified + 94 Proposal + 161 Won + 183 Lost; 144 won+matched / 140 with orders; 3,530 attributed orders (3,147 net) / 10,551,748.55 gross / 9,766,828.34 net; 8/8 full matches via independent recompute. Gross fix 2026-11-20: previously SUM(line_total) without order_date_valid = 10,912,463.13, including the matched companies' 330 invalid-date lines worth 360,714.58; now the KPI 1 valid-date basis, exact tie with kpi_12_lead_score_vs_order_value.'
)
AS
WITH orders_per_company AS (
  SELECT
    company_sk,
    COUNT(DISTINCT order_id) AS order_count,
    COUNT(DISTINCT CASE WHEN status <> 'cancelled' AND order_date_valid
                        THEN order_id END) AS net_order_count,
    -- KPI 1 gross basis: ALL statuses, valid order dates only
    ROUND(SUM(CASE WHEN order_date_valid THEN line_total ELSE 0 END), 2) AS gross_revenue,
    ROUND(SUM(CASE WHEN status <> 'cancelled' AND order_date_valid
                   THEN line_total ELSE 0 END), 2) AS net_revenue
  FROM b2b_ecommerce.gold.fact_order_items
  GROUP BY company_sk
),
won AS (
  SELECT source, campaign_name, company_sk
  FROM b2b_ecommerce.gold.fact_leads
  WHERE status = 'Won' AND company_sk IS NOT NULL
),
won_lead_counts AS (
  SELECT source, campaign_name, COUNT(*) AS won_matched_lead_count
  FROM won
  GROUP BY source, campaign_name
),
won_leads_with_orders AS (
  SELECT w.source, w.campaign_name, COUNT(*) AS won_leads_with_orders_count
  FROM won w
  JOIN orders_per_company o
    ON w.company_sk = o.company_sk
  GROUP BY w.source, w.campaign_name
),
company_attr AS (
  SELECT
    wc.source,
    wc.campaign_name,
    COALESCE(SUM(o.order_count), 0) AS attributed_order_count,
    COALESCE(SUM(o.net_order_count), 0) AS attributed_net_order_count,
    COALESCE(SUM(o.gross_revenue), 0) AS attributed_gross_revenue,
    COALESCE(SUM(o.net_revenue), 0) AS attributed_net_revenue
  FROM (
    SELECT DISTINCT source, campaign_name, company_sk
    FROM won
  ) wc
  LEFT JOIN orders_per_company o
    ON wc.company_sk = o.company_sk
  GROUP BY wc.source, wc.campaign_name
),
per_campaign AS (
  SELECT
    source,
    campaign_name,
    MAX(utm_source) AS utm_source,
    MAX(utm_medium) AS utm_medium,
    MAX(utm_campaign) AS utm_campaign,
    COUNT(*) AS lead_count,
    SUM(CASE WHEN status = 'New' THEN 1 ELSE 0 END) AS new_lead_count,
    SUM(CASE WHEN status = 'Contacted' THEN 1 ELSE 0 END) AS contacted_lead_count,
    SUM(CASE WHEN status = 'Qualified' THEN 1 ELSE 0 END) AS qualified_lead_count,
    SUM(CASE WHEN status = 'Proposal' THEN 1 ELSE 0 END) AS proposal_lead_count,
    SUM(CASE WHEN status = 'Won' THEN 1 ELSE 0 END) AS won_lead_count,
    SUM(CASE WHEN status = 'Lost' THEN 1 ELSE 0 END) AS lost_lead_count
  FROM b2b_ecommerce.gold.fact_leads
  GROUP BY source, campaign_name
),
joined AS (
  SELECT
    p.source,
    p.campaign_name,
    p.utm_source,
    p.utm_medium,
    p.utm_campaign,
    p.lead_count,
    p.new_lead_count,
    p.contacted_lead_count,
    p.qualified_lead_count,
    p.proposal_lead_count,
    p.won_lead_count,
    p.lost_lead_count,
    COALESCE(lc.won_matched_lead_count, 0) AS won_matched_lead_count,
    COALESCE(lo.won_leads_with_orders_count, 0) AS won_leads_with_orders_count,
    COALESCE(ca.attributed_order_count, 0) AS attributed_order_count,
    COALESCE(ca.attributed_net_order_count, 0) AS attributed_net_order_count,
    COALESCE(ca.attributed_gross_revenue, 0) AS attributed_gross_revenue,
    COALESCE(ca.attributed_net_revenue, 0) AS attributed_net_revenue
  FROM per_campaign p
  LEFT JOIN won_lead_counts lc
    ON p.source = lc.source AND p.campaign_name = lc.campaign_name
  LEFT JOIN won_leads_with_orders lo
    ON p.source = lo.source AND p.campaign_name = lo.campaign_name
  LEFT JOIN company_attr ca
    ON p.source = ca.source AND p.campaign_name = ca.campaign_name
)
SELECT
  source,
  campaign_name,
  utm_source,
  utm_medium,
  utm_campaign,
  lead_count,
  new_lead_count,
  contacted_lead_count,
  qualified_lead_count,
  proposal_lead_count,
  won_lead_count,
  lost_lead_count,
  ROUND(100 * qualified_lead_count / lead_count, 2) AS qualified_rate_pct,
  ROUND(100 * proposal_lead_count / lead_count, 2) AS proposal_rate_pct,
  ROUND(100 * won_lead_count / lead_count, 2) AS won_rate_pct,
  ROUND(100 * lost_lead_count / lead_count, 2) AS lost_rate_pct,
  CASE WHEN qualified_lead_count > 0
       THEN ROUND(100 * won_lead_count / qualified_lead_count, 2)
  END AS won_over_qualified_pct,
  won_matched_lead_count,
  won_leads_with_orders_count,
  attributed_order_count,
  attributed_net_order_count,
  attributed_gross_revenue,
  attributed_net_revenue,
  CASE WHEN won_matched_lead_count > 0
       THEN ROUND(attributed_net_revenue / won_matched_lead_count, 2)
  END AS attributed_net_revenue_per_won_lead,
  ROUND(100 * lead_count / SUM(lead_count) OVER (), 2) AS lead_share_pct,
  ROUND(100 * attributed_net_revenue / SUM(attributed_net_revenue) OVER (), 2) AS attributed_net_revenue_share_pct,
  ROW_NUMBER() OVER (
    ORDER BY attributed_net_revenue DESC, source ASC, campaign_name ASC
  ) AS attributed_net_revenue_rank
FROM joined

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 8 rows = 8 distinct
-- (source, campaign_name) pairs; SUM(lead_count) = 1,098 with the 6 status
-- counts summing exactly (285 New / 237 Contacted / 138 Qualified / 94
-- Proposal / 161 Won / 183 Lost); SUM(won_matched_lead_count) = 144 and
-- SUM(won_leads_with_orders_count) = 140 (4 matched companies have no
-- orders); SUM(attributed_order_count) = 3,530 / net 3,147 = the 144 matched
-- companies' full order history; SUM(attributed_gross_revenue) = 10,551,748.55 (KPI 1 valid-date
-- basis; fix 2026-11-20 - the pre-fix 10,912,463.13 unfiltered SUM included
-- 330 invalid-date lines worth 360,714.58; gross now ties exactly to
-- kpi_12_lead_score_vs_order_value) and SUM(attributed_net_revenue) =
-- 9,766,828.34 (unchanged by the fix - no double attribution, companies
-- are in exactly one campaign); status counts sum to lead_count on every row; rates recompute
-- exactly; share columns sum ~100.00 (per-row ROUND); rank contiguous 1..8
-- and identical to a fresh recompute; 0 null keys; utm maps 1:1 per campaign.
WITH ranked AS (
  SELECT v.*,
         ROW_NUMBER() OVER (
           ORDER BY attributed_net_revenue DESC, source ASC, campaign_name ASC
         ) AS rank_recompute
  FROM b2b_ecommerce.gold.kpi_11_lead_conversion_funnel v
),
utm_cardinality AS (
  SELECT COUNT(*) AS campaigns_with_multi_utm
  FROM (
    SELECT source, campaign_name,
           COUNT(DISTINCT utm_source || '|' || utm_medium || '|' || utm_campaign) AS utm_variants
    FROM b2b_ecommerce.gold.fact_leads
    GROUP BY source, campaign_name
    HAVING COUNT(DISTINCT utm_source || '|' || utm_medium || '|' || utm_campaign) > 1
  )
)
SELECT
  COUNT(*) AS rows_source_campaign,
  COUNT(DISTINCT source || '|' || campaign_name) AS distinct_pairs,
  COUNT(DISTINCT source) AS distinct_sources,
  SUM(lead_count) AS total_leads,
  SUM(new_lead_count) AS total_new,
  SUM(contacted_lead_count) AS total_contacted,
  SUM(qualified_lead_count) AS total_qualified,
  SUM(proposal_lead_count) AS total_proposal,
  SUM(won_lead_count) AS total_won,
  SUM(lost_lead_count) AS total_lost,
  SUM(won_matched_lead_count) AS total_won_matched,
  SUM(won_leads_with_orders_count) AS total_won_leads_with_orders,
  SUM(attributed_order_count) AS total_attributed_orders,
  SUM(attributed_net_order_count) AS total_attributed_net_orders,
  SUM(attributed_gross_revenue) AS total_attributed_gross_revenue,
  SUM(attributed_net_revenue) AS total_attributed_net_revenue,
  SUM(CASE WHEN new_lead_count + contacted_lead_count + qualified_lead_count
              + proposal_lead_count + won_lead_count + lost_lead_count <> lead_count
           THEN 1 ELSE 0 END) AS decomposition_mismatches,
  SUM(CASE WHEN won_matched_lead_count > won_lead_count
                OR won_leads_with_orders_count > won_matched_lead_count
           THEN 1 ELSE 0 END) AS attribution_invariants,
  SUM(CASE WHEN qualified_rate_pct <> ROUND(100 * qualified_lead_count / lead_count, 2)
                OR won_rate_pct <> ROUND(100 * won_lead_count / lead_count, 2)
                OR lost_rate_pct <> ROUND(100 * lost_lead_count / lead_count, 2)
                OR proposal_rate_pct <> ROUND(100 * proposal_lead_count / lead_count, 2)
           THEN 1 ELSE 0 END) AS rate_mismatches,
  SUM(CASE WHEN qualified_lead_count > 0
            AND won_over_qualified_pct <> ROUND(100 * won_lead_count / qualified_lead_count, 2)
           THEN 1 ELSE 0 END) AS stage_rate_mismatches,
  ROUND(SUM(lead_share_pct), 2) AS lead_share_sum,
  ROUND(SUM(attributed_net_revenue_share_pct), 2) AS revenue_share_sum,
  COUNT(DISTINCT attributed_net_revenue_rank) AS distinct_ranks,
  MAX(attributed_net_revenue_rank) AS max_rank,
  SUM(CASE WHEN attributed_net_revenue_rank <> rank_recompute
           THEN 1 ELSE 0 END) AS rank_mismatches,
  SUM(CASE WHEN source IS NULL OR campaign_name IS NULL OR utm_source IS NULL
                OR utm_medium IS NULL OR utm_campaign IS NULL
           THEN 1 ELSE 0 END) AS null_keys,
  (SELECT campaigns_with_multi_utm FROM utm_cardinality) AS campaigns_with_multi_utm,
  SUM(CASE WHEN won_matched_lead_count > 0
            AND attributed_net_revenue_per_won_lead
                <> ROUND(attributed_net_revenue / won_matched_lead_count, 2)
           THEN 1 ELSE 0 END) AS per_won_lead_mismatches
FROM ranked

In [0]:
%sql
-- Verification: independently recompute every campaign cell through a
-- different path - funnel counts via SUM(CAST(boolean AS INT)) conditional
-- aggregation plus an EXISTS subquery for the orders bridge (vs the view's
-- CASE WHEN + pre-aggregated orders CTE joins), attribution through
-- company-grain DISTINCT (campaign, company) joined once to order lines
-- (vs the view's orders-per-company pre-aggregation) - then FULL OUTER JOIN
-- against the stored view and compare the count and revenue measures.
-- Expected: 8 rows compared, 8/8 full matches, 0 rows only on either side.
-- Fix note 2026-11-20: this recompute originally shared the view's missing
-- order_date_valid filter on gross (same bug on both paths, so the pre-fix
-- compare passed) - the gross line below now filters identically to the view.
-- NOTE: COUNTIF is not resolvable in this SQL path and SUM(boolean) is
-- rejected by strict typing - SUM(CAST(cond AS INT)) is the working form.
WITH per_campaign_recomputed AS (
  SELECT
    source,
    campaign_name,
    COUNT(*) AS lead_count,
    SUM(CAST(status = 'New' AS INT)) AS new_lead_count,
    SUM(CAST(status = 'Contacted' AS INT)) AS contacted_lead_count,
    SUM(CAST(status = 'Qualified' AS INT)) AS qualified_lead_count,
    SUM(CAST(status = 'Proposal' AS INT)) AS proposal_lead_count,
    SUM(CAST(status = 'Won' AS INT)) AS won_lead_count,
    SUM(CAST(status = 'Lost' AS INT)) AS lost_lead_count,
    SUM(CAST(status = 'Won' AND company_sk IS NOT NULL AS INT)) AS won_matched_lead_count,
    SUM(CAST(status = 'Won' AND company_sk IS NOT NULL
            AND EXISTS (
              SELECT 1 FROM b2b_ecommerce.gold.fact_order_items oi
              WHERE oi.company_sk = l.company_sk
            ) AS INT)) AS won_leads_with_orders_count
  FROM b2b_ecommerce.gold.fact_leads l
  GROUP BY source, campaign_name
),
revenue_recomputed AS (
  -- company-grain: DISTINCT (campaign, company) then join order lines once;
  -- fan-out-free because each matched company has exactly one Won lead
  -- (verified 144 = 144) - independent of the view's pre-aggregation path.
  SELECT
    c.source,
    c.campaign_name,
    COUNT(DISTINCT o.order_id) AS attributed_order_count,
    COUNT(DISTINCT CASE WHEN o.status <> 'cancelled' AND o.order_date_valid
                        THEN o.order_id END) AS attributed_net_order_count,
    ROUND(SUM(CASE WHEN o.order_date_valid THEN o.line_total ELSE 0 END), 2)
      AS attributed_gross_revenue,
    ROUND(SUM(CASE WHEN o.status <> 'cancelled' AND o.order_date_valid
                   THEN o.line_total ELSE 0 END), 2) AS attributed_net_revenue
  FROM (
    SELECT DISTINCT source, campaign_name, company_sk
    FROM b2b_ecommerce.gold.fact_leads
    WHERE status = 'Won' AND company_sk IS NOT NULL
  ) c
  JOIN b2b_ecommerce.gold.fact_order_items o
    ON c.company_sk = o.company_sk
  GROUP BY c.source, c.campaign_name
),
recomputed AS (
  SELECT p.*, COALESCE(r.attributed_order_count, 0) AS attributed_order_count,
         COALESCE(r.attributed_net_order_count, 0) AS attributed_net_order_count,
         COALESCE(r.attributed_gross_revenue, 0) AS attributed_gross_revenue,
         COALESCE(r.attributed_net_revenue, 0) AS attributed_net_revenue
  FROM per_campaign_recomputed p
  LEFT JOIN revenue_recomputed r
    ON p.source = r.source AND p.campaign_name = r.campaign_name
),
compare AS (
  SELECT v.source AS v_source, v.campaign_name AS v_campaign,
         v.lead_count AS v_leads, v.qualified_lead_count AS v_qualified,
         v.won_lead_count AS v_won, v.lost_lead_count AS v_lost,
         v.won_matched_lead_count AS v_won_matched,
         v.won_leads_with_orders_count AS v_won_with_orders,
         v.attributed_order_count AS v_orders, v.attributed_net_order_count AS v_net_orders,
         v.attributed_gross_revenue AS v_gross, v.attributed_net_revenue AS v_net,
         r.source AS r_source, r.campaign_name AS r_campaign,
         r.lead_count AS r_leads, r.qualified_lead_count AS r_qualified,
         r.won_lead_count AS r_won, r.lost_lead_count AS r_lost,
         r.won_matched_lead_count AS r_won_matched,
         r.won_leads_with_orders_count AS r_won_with_orders,
         r.attributed_order_count AS r_orders, r.attributed_net_order_count AS r_net_orders,
         r.attributed_gross_revenue AS r_gross, r.attributed_net_revenue AS r_net
  FROM b2b_ecommerce.gold.kpi_11_lead_conversion_funnel v
  FULL OUTER JOIN recomputed r
    ON v.source = r.source AND v.campaign_name = r.campaign_name
)
SELECT
  COUNT(*) AS rows_compared,
  SUM(CASE WHEN v_source IS NOT NULL AND r_source IS NOT NULL
            AND v_leads = r_leads AND v_qualified = r_qualified
            AND v_won = r_won AND v_lost = r_lost
            AND v_won_matched = r_won_matched
            AND v_won_with_orders = r_won_with_orders
            AND v_orders = r_orders AND v_net_orders = r_net_orders
            AND v_gross = r_gross AND v_net = r_net
           THEN 1 ELSE 0 END) AS full_matches,
  SUM(CASE WHEN v_source IS NULL THEN 1 ELSE 0 END) AS view_only_rows,
  SUM(CASE WHEN r_source IS NULL THEN 1 ELSE 0 END) AS recompute_only_rows
FROM compare

In [0]:
%sql
-- Headline: all 8 campaigns by attributed net revenue rank.
SELECT attributed_net_revenue_rank, source, campaign_name, utm_medium,
       lead_count, qualified_lead_count, won_lead_count, won_rate_pct,
       won_over_qualified_pct, lost_lead_count,
       won_matched_lead_count, won_leads_with_orders_count,
       attributed_order_count, attributed_net_order_count,
       attributed_gross_revenue, attributed_net_revenue,
       attributed_net_revenue_per_won_lead, attributed_net_revenue_share_pct
FROM b2b_ecommerce.gold.kpi_11_lead_conversion_funnel
ORDER BY attributed_net_revenue_rank